# Lantern LAGN First Cut Filter

*Based on notebook written by Zach Gillis, available at https://github.com/drphilmarshall/lantern/tree/main/notebooks*

*This revised notebook is written by Natascha Barac*

*Date last updated: 28 May 2026*

This notebook prepares our *First Cut* filter, which selects lensed-AGN (LAGN) targets from the Rubin Observatory alert stream using difference image analysis sources (DIASources). The trained filter is implemented at ANTARES, and sources which pass the filter there are tagged as lantern targets (https://antares.noirlab.edu/tags). This notebook is meant to be run on the cloud-based Rubin Science Platform JupyterLab server. (This was last tested on RSP release 29.2.0).

## Overview

Rubin's difference imaging pipeline subtracts a template image from each visit image, producing a "difference image" where only flux changes appear. Sources detected in these difference images are called DIASources. The full DP1 dataset contains 1.4 million DIASources within our three target survey fields. Our goal is to progressively filter this sample down to sources consistent with spatially extended flux differences that could be lensed AGN.

Lensed AGN are predicted to appear and be efficiently detected as spatially extended sources in difference images from optical imaging surveys (Kochanek et al. 2006). This motivates our use of extendedness metrics as the primary discriminating features.

## Dataset

Our dataset (most recently `combined_training_data_v2.0.7`) is built using the DIASources of simulated LAGN injected in the ECDFS field, which are assigned ground-truth `lagn` labels. Each LAGN injected is assigned a unique `lens_id` value so that we can identify unique lens systems.

Our non-LAGN DIASources are the full set of DP1 DIASources in the ECDFS field. We use only ECDFS currently because our injections are only in the ECDFS field. Note that we assume all of these sources are non-LAGN. We also do not yet include imposters in our training set.

| Field | RA | Dec | Notes |
|---|---|---|---|
| ECDFS | 53.16° | −28.10° | Extended Chandra Deep Field South |

Non-LAGN have been clustered into sky positions of 3arcsec, as seen in the data preparation notebook. Each sky position has been assigned a unique `lens_id` as well, so that we can stratify the training and test sets based on unique sources. Non-LAGN can be distinguished from LAGN because they are assigne negative `lens_id` values.

## Model

This notebook trains an XGBoost binary classifier to distinguish LAGN DIASources from non-LAGN DIASources, evaluates it on a held-out test set, and prepares a completeness/purity curve using estimated LSST survey statistics.

## Summary

1. **Load data** — read in combined_training_data_v{version}.csv and split into stratified training and test sets, making sure to keep unique lenses in a single set to prevent data leakage
2. **Hyperparameter search** — 5-fold stratified CV with `GridSearchCV` over key XGBoost parameters, scored by ROC-AUC, which is invariant to class ratio
3. **Train final model** — fit XGBoost with best parameters and evaluate on held-out test set; includes corner plot
4. **Survey Scale Analysis** — project to true class ratios and survey scale

---

## 1. Imports & Data Loading

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yaml
import corner

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    roc_curve, auc, confusion_matrix,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
)

import data_processing as dp

from sklearn.calibration import calibration_curve

### Load & Split Data

Loads `combined_training_data_{version}.csv` — which contains one row per DIASource with `label=0` (non-LAGN DIASource, DP1) or `label=1` (LAGN DIASource, injected).

A subset of features is selected from the candidate features in `combined_training_data_{version}.csv`. Currently, several features are commented out to reduce overfitting risk (`centroid_flag`, `trailLength`, `trailFlux`, and the dipole error/flux columns). A next step in this work is to explore a broader variety of features.

The dataset is split 80% train / 20% test with `stratify=y` to preserve the class ratio in both splits.

#### Band Weighting

The injected LAGN (label=1) and DP1 background (label=0) have different band distributions. Without correction, the model could implicitly learn band membership as a proxy for class label rather than the underlying physical features. To address this, each training sample is assigned a weight inversely proportional to the count of its `(label, band)` cell, normalized so the mean weight is 1. 

This ensures every `(label, band)` combination contributes equally to the loss. Because LAGN cells are ~400–800× smaller than non-LAGN cells, these weights also subsume the overall class imbalance correction. Weights are passed to `model.fit()` via `sample_weight` and recomputed per fold in the Section 5 cross-validation.

XGBoost natively supports NaN values (present in dipole-specific columns in DIASources without attempted dipole fits) and categorical variables (i.e. `band`).

**Active features:** `band`, `psf_fwhm`, `snr`, `template_flux`, `scienceFlux`, `psfFlux`, `apFlux`, `temp_sci_flux_ratio`, `moment_ext`, `ellip_ext`, `flux_ext`, `extendedness`, `psfChi2`, `dipoleFitAttempted`, `dipoleChi2`, `dipoleLength`, `x_y_err`

In [ ]:
training_data = 'combined_training_data_v2.0.7.csv'

FEATURES = [
    # Other
    'band',
    # 'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    # 'trailFlux',
    # 'trailLength',
    
    # Dipole
    # 'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    # 'dipoleFluxDiffErr',
    # 'dipoleMeanFlux',
    # 'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

df = pd.read_csv(training_data)

df['isDipole'] = df['isDipole'] == 'True'
df['band'] = pd.Categorical(df['band'])
band_categories = df['band'].cat.categories

We use `StratifiedGroupKFold` to prevent data leakage (keep unique lenses in the same subset), and preserve the class distribution in each fold. 

In [ ]:
X = df.drop(columns=['label', 'lens_id'])[FEATURES]
y = df['label']
groups = df['lens_id']

# Split by group using StratifiedGroupKFold: 80% train_full, 20% test
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_full_idx, test_idx = next(sgkf.split(X, y, groups=groups))

# Second split: from train_full, 80% train, 20% eval
X_train_full = X.iloc[train_full_idx]
y_train_full = y.iloc[train_full_idx]
groups_train_full = groups.iloc[train_full_idx]

sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=43)
train_idx_temp, eval_idx_temp = next(sgkf2.split(X_train_full, y_train_full, groups=groups_train_full))

# Map back to original indices
train_idx = train_full_idx[train_idx_temp]
eval_idx = train_full_idx[eval_idx_temp]

# Create final subsets
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_eval, y_eval = X.iloc[eval_idx], y.iloc[eval_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# Get groups for each split
groups_train = groups.iloc[train_idx]
groups_eval = groups.iloc[eval_idx]
groups_test = groups.iloc[test_idx]

print(f"\nSplits:")
print(f"  Train: {len(X_train):,} DIAsources, {groups_train[y_train==1].nunique():,} lenses, {groups_train[y_train==0].nunique():,} non-lenses")
print(f"  Eval:  {len(X_eval):,} DIAsources, {groups_eval[y_eval==1].nunique():,} lenses, {groups_eval[y_eval==0].nunique():,} non-lenses")
print(f"  Test:  {len(X_test):,} DIAsources, {groups_test[y_test==1].nunique():,} lenses, {groups_test[y_test==0].nunique():,} non-lenses")

In [ ]:
# Compute sample weights
def compute_band_weights(y_s, X_s):
    tmp = pd.DataFrame({'label': y_s.values, 'band': X_s['band'].values})
    cell_counts = tmp.groupby(['label', 'band'], observed=True)['label'].transform('count')
    w = 1.0 / cell_counts
    return (w / w.mean()).values

sample_weight_train = compute_band_weights(y_train, X_train)

print("\nTraining samples by (label, band):")
print(pd.crosstab(y_train, X_train['band']))

tmp = pd.DataFrame({'label': y_train.values, 'band': X_train['band'].values, 'weight': sample_weight_train})
print("\nMean sample weight by (label, band):")
print(tmp.groupby(['label', 'band'], observed=True)['weight'].mean().unstack(fill_value=0).round(3))

___
## 2. Hyperparameter Search

### Grid Search with 5-fold Cross-Validation

Searches over a grid of XGBoost hyperparameters using `GridSearchCV` with 5-fold stratified cross-validation, scored by ROC-AUC.

ROC-AUC is used because it is invariant to class ratio — this data has a lens : non-lens ratio of ~1:50, compared to the expected survey ratio of 1 : 37,000.

There are CV options other than grid search; this is the simplest but takes the most amount of time. 

This cell takes over an hour to run. You can skip this step if wanting to use the hyperparameters that have already been optimized. The next code block manually sets parameters. 

In [ ]:
param_grid = {
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.01, 0.02, 0.03],
    'subsample':        [0.6, 0.8],
    'colsample_bytree': [0.7, 0.9],
}

search = GridSearchCV(
    XGBClassifier(
        n_estimators=500,
        # scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    refit=False,
    verbose=1,
)
search.fit(X_train, y_train, sample_weight=sample_weight_train)

best_params = search.best_params_
print(f"\nBest params: {best_params}")

___
## 3. Train & Evaluate Model

We train an XGBoost model using the best hyperparameters from the grid_search (or set `USE_BEST_PARAMS = False` if you'd like to set your own parameters manually). The manual parameters here are the best hyperparameters from a previous run.

We then evaluate on a held-out test set. Note that the precision and recall are skewed by the large class imbalance. 

In [ ]:
USE_BEST_PARAMS = False

MANUAL_PARAMS = {
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.9,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, sample_weight=sample_weight_train,
          eval_set=[(X_eval, y_eval)], verbose=False)
print(f"Training complete. Best n_estimators: {model.best_iteration}")

Now we evaluate test set metrics using the held-out test sets, X_test and y_test. 

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-LAGN (0)', 'LAGN (1)'], digits=4))

### Visualization: Diagnostic Plots

Five diagnostic plots summarizing classifier performance on the held-out test set:

| Plot | Description |
|---|---|
| **Feature Importance** | XGBoost gain-based importance for each input feature |
| **ROC Curve** | True positive rate vs. false positive rate across all thresholds |
| **Precision-Recall Curve** | Precision vs. recall at the training-set class ratio (~1:50) |
| **P(LAGN) Distribution** | Predicted probability histograms for LAGN and non-LAGN DIASources |
| **Confusion Matrix** | Counts and row-normalized rates at nominal threshold (0.5) |

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# Feature Importance
ax1 = fig.add_subplot(gs[0, 0])
importance = model.feature_importances_
feat_names = np.array(X.columns.tolist())
top_idx = np.argsort(importance)[-20:]
ax1.barh(feat_names[top_idx], importance[top_idx], color='steelblue', edgecolor='white')
ax1.set_xlabel('Gain', fontsize=11)
ax1.set_title('Feature Importance', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelsize=8)
ax1.spines[['top', 'right']].set_visible(False)

# ROC Curve
ax2 = fig.add_subplot(gs[0, 1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost\n(AUC = {roc_auc_val:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random')
ax2.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.02])
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)

# Precision-Recall Curve
ax3 = fig.add_subplot(gs[0, 2])
pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
baseline = y_test.mean()
ax3.plot(pr_recall, pr_precision, color='darkorange', lw=2, label=f'XGBoost\n(AP = {ap:.4f})')
ax3.axhline(baseline, color='navy', lw=1.5, linestyle='--',
            label=f'Random\n(AP = {baseline:.4f})')
ax3.fill_between(pr_recall, pr_precision, alpha=0.08, color='darkorange')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.02])
ax3.set_xlabel('Recall', fontsize=11)
ax3.set_ylabel('Precision', fontsize=11)
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.spines[['top', 'right']].set_visible(False)

# Predicted Probability Distribution
ax4 = fig.add_subplot(gs[1, 0])
bins = np.linspace(0, 1, 60)
ax4.hist(y_prob[y_test == 0], bins=bins, alpha=0.65, color='steelblue',
         label='Non-LAGN (0)', density=True)
ax4.hist(y_prob[y_test == 1], bins=bins, alpha=0.65, color='darkorange',
         label='LAGN (1)', density=True)
ax4.axvline(0.5, color='red', linestyle='--', lw=1.5, label='Threshold = 0.5')
ax4.set_xlabel('P(LAGN)', fontsize=11)
ax4.set_ylabel('Density', fontsize=11)
ax4.set_title('P(LAGN)', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10, loc='upper center')
ax4.spines[['top', 'right']].set_visible(False)

# Confusion Matrix Heatmap
ax5 = fig.add_subplot(gs[1, 1])
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels_raw = [f'{v:,}' for v in cm.ravel()]
labels_pct = [f'{v:.1%}' for v in cm_norm.ravel()]
annot = np.array([f'{r}\n({p})' for r, p in zip(labels_raw, labels_pct)]).reshape(2, 2)

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax5,
            xticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            yticklabels=['Non-LAGN (0)', 'LAGN (1)'],
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Row-normalised rate'})
ax5.set_xlabel('Predicted Label', fontsize=11)
ax5.set_ylabel('True Label', fontsize=11)
ax5.set_title('Confusion Matrix', fontsize=12, fontweight='bold')

plt.show()          

In [ ]:
# Print feature importance
sorted_idx = np.argsort(importance)[::-1]
print('Full feature importance:')
for rank, i in enumerate(sorted_idx, 1):
    print(f"{rank:>3}. {feat_names[i]:<35} {importance[i]:.6f}")                                      

In [ ]:
# SELECT FEATURES TO PLOT
SELECTED_FEATURES = [
    'snr',
    'x_y_err',
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'dipoleChi2',
    'dipoleLength',
    # Add or remove features as desired
]

FEATURE_PROPS = {
    'psf_fwhm':            ('linear', (0, 1),        'PSF FWHM'),
    'snr':                 ('linear', (0, 50),         'SNR'),
    'template_flux':       ('log',    (1e2, 1e7),      'Temp. Flux'),
    'scienceFlux':         ('log',    (1e2, 1e7),      'Sci. Flux'),
    'psfFlux':             ('log',    (1e2, 1e7),      'PSF Flux'),
    'apFlux':              ('log',    (1e2, 1e7),      'Ap. Flux'),
    'temp_sci_flux_ratio': ('linear', (0, 2),          'Temp./Sci.\nFlux'),
    'moment_ext':          ('linear', (0, 4),          'Moment Ext.'),
    'ellip_ext':           ('linear', (0, 1),          'Ellip. Diff.'),
    'flux_ext':            ('log',    (0.1, 10.0),     'Flux Ext.'),
    'extendedness':        ('linear', (0, 1),          'Rubin Ext.'),
    'psfChi2':             ('log',    (1, 1e5),        'PSF χ²'),
    'isDipole':            ('linear', (-0.1, 1.1),     'Is Dipole'),
    'dipoleFitAttempted':  ('linear', (-0.1, 1.1),     'Dip. Fit?'),
    'dipoleChi2':          ('log',    (1e2, 1e5),     'Dip. χ²'),
    'dipoleLength':        ('linear', (-0.1, 0.15),        'Dip. Length'),
    'x_y_err':             ('linear', (-1, 4),          'Cent. Err.'),
    'trailLength':         ('linear', (0, 4),          'Trail Length'),
    'trailFlux':           ('log',    (1e2, 1e7),      'Trail Flux'),
    'dipoleMeanFlux':      ('log',    (1e2, 1e7),      'Dipole Mean Flux'),
    'dipoleFluxDiffErr':   ('log',    (1e0, 1e5),      'Dipole Flux Diff Err'),
    'dipoleMeanFluxErr':   ('log',    (1e0, 1e5),      'Dipole Mean Flux Err'),
    'centroid_flag':       ('linear', (-0.1, 1.1),     'Centroid Flag'),
}

def to_array(subset, cols):
    arr = np.empty((len(subset), len(cols)), dtype=float)
    for i, c in enumerate(cols):
        scale, (lo, hi) = FEATURE_PROPS[c][0], FEATURE_PROPS[c][1]
        s = pd.to_numeric(subset[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if scale == 'log':
            s = s.where(s > 0)      # mask non-positive: log(≤0) is undefined
        med = s.median()
        fill = med if np.isfinite(med) else lo   # fall back to range lower bound
        s = s.fillna(fill)
        arr[:, i] = s.values
        # Final safety: clamp any surviving bad values
        if scale == 'log':
            bad = ~np.isfinite(arr[:, i]) | (arr[:, i] <= 0)
            arr[bad, i] = lo
        else:
            arr[~np.isfinite(arr[:, i]), i] = 0.0
    return arr

df_false = df[df['label'] == 0]
df_true  = df[df['label'] == 1]

XGB_THRESHOLD = 0.9691 #this is for 95% completeness, calculated from "Survey-Scale Analysis" below. 
all_probs = model.predict_proba(X)[:, 1]
df_xgb = df[all_probs > XGB_THRESHOLD]
print(f"Sources passing XGBoost (threshold={XGB_THRESHOLD}): {len(df_xgb):,}")

# Filter candidate_cols to only include SELECTED_FEATURES
candidate_cols = [f for f in SELECTED_FEATURES if f in FEATURES and f != 'band']
print(f"Using {len(candidate_cols)} selected features: {candidate_cols}")

# Drop columns where any population has fewer than 2 unique in-range values
def is_plottable(col, *dfs, min_samples=2):
    scale, (lo, hi) = FEATURE_PROPS[col][0], FEATURE_PROPS[col][1]
    for d in dfs:
        s = pd.to_numeric(d[col], errors='coerce')
        if scale == 'log':
            s = s.where(s > 0)
        vals = s.dropna()
        in_range = vals[(vals >= lo) & (vals <= hi)]
        if in_range.nunique() < min_samples:
            return False
    return True

columns_corner = [c for c in candidate_cols if is_plottable(c, df_false, df_true, df_xgb)]
dropped = set(candidate_cols) - set(columns_corner)
if dropped:
    print(f"Dropped unplottable columns: {dropped}")

print(f"Final plotting columns ({len(columns_corner)}): {columns_corner}")

axes_scale = [FEATURE_PROPS[f][0] for f in columns_corner]
ranges     = [FEATURE_PROPS[f][1] for f in columns_corner]
labels     = [FEATURE_PROPS[f][2] for f in columns_corner]

data_array_all  = to_array(df_false, columns_corner)
data_array_lagn = to_array(df_true,  columns_corner)
data_array_xgb  = to_array(df_xgb,   columns_corner)

# Adjust figure size based on number of features
n_features = len(columns_corner)
fig_size = max(8, min(15, n_features * 2.5))  # Scale between 8 and 15

fig = corner.corner(data_array_all,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=plt.figure(figsize=(fig_size, fig_size), dpi=300),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_lagn,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='green',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

fig = corner.corner(data_array_xgb,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='red',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                   )

for ax in fig.axes:
    ax.tick_params(labelsize=8, axis='both', which='major', pad=2)
    for label in ax.get_xticklabels():
        label.set_rotation(0)
    xlabels = [t for t in ax.xaxis.get_major_ticks() if t.label1.get_visible()]
    if xlabels:
        xlabels[-1].label1.set_visible(False)
    ylabels = [t for t in ax.yaxis.get_major_ticks() if t.label1.get_visible()]
    if ylabels:
        ylabels[0].label1.set_visible(False)

equation_text = (
    r'$\mathrm{Flux\ Ext.} = \dfrac{F_\mathrm{ap}}{F_\mathrm{PSF}}$' + '\n\n' +
    r'$\mathrm{Moment\ Ext.} = \dfrac{I_{xx} + I_{yy}}{I_{xx}^{\,\mathrm{PSF}} + I_{yy}^{\,\mathrm{PSF}}}$' + '\n\n' +
    r'$\mathrm{Ellip.\ Diff.} = \dfrac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - '
    r'\dfrac{\sqrt{(I_{xx}^{\,\mathrm{PSF}}-I_{yy}^{\,\mathrm{PSF}})^2+4(I_{xy}^{\,\mathrm{PSF}})^2}}'
    r'{I_{xx}^{\,\mathrm{PSF}}+I_{yy}^{\,\mathrm{PSF}}}$'
)

# fig.text(0.55, 0.87, equation_text, fontsize=13,
#          bbox=dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='white', alpha=0),
#          verticalalignment='top')

legend_elements = [
    Patch(facecolor='grey',  edgecolor='black',    label=f'All DP1 DIASources ({len(df_false):,})'),
    Patch(facecolor='green', edgecolor='darkgreen', label=f'Injected LAGN ({len(df_true):,})'),
    Patch(facecolor='red',   edgecolor='darkred',   label=f'XGBoost p > {XGB_THRESHOLD} ({len(df_xgb):,})'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=13, framealpha=0.9)

plt.show()

### Train & Save Final Model
We find the best n_estimators using early stopping, and then train a final model using all of the available data. This way, our model has seen as many lenses as possible when it gets applied to real data at ANTARES. Note that we did not do this during testing above, to avoid data leakage and make sure that we were testing on a set that the model had not seen before.

In [ ]:
# ── Train Final Model ──────────────────────────────────────────────────────────
USE_BEST_PARAMS = False

MANUAL_PARAMS = {
    'max_depth':        8,
    'learning_rate':    0.03,
    'subsample':        0.6,
    'colsample_bytree': 0.9,
}

params = best_params if USE_BEST_PARAMS else MANUAL_PARAMS

# Train with early stopping to find best n_estimators
print("Training with early stopping to find optimal n_estimators...")
model = XGBClassifier(
    n_estimators=500,
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    **params,
)
model.fit(X_train, y_train, sample_weight=sample_weight_train,
          eval_set=[(X_eval, y_eval)], verbose=False)
best_n_estimators = model.best_iteration
print(f"Best n_estimators: {best_n_estimators}")

# Retrain on train+eval+test with the optimal n_estimators (no early stopping)
print(f"\nRetraining final model on train+eval+test data with n_estimators={best_n_estimators}...")
X_train_final = pd.concat([X_train, X_eval, X_test])
y_train_final = pd.concat([y_train, y_eval, y_test])
sample_weight_eval = compute_band_weights(y_eval, X_eval)
sample_weight_test = compute_band_weights(y_test, X_test)
sample_weight_final = np.concatenate([sample_weight_train, sample_weight_eval, sample_weight_test])

final_model = XGBClassifier(
    n_estimators=best_n_estimators,  # Use the optimal value found
    eval_metric='logloss',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    **params,
)

final_model.fit(X_train_final, y_train_final, sample_weight=sample_weight_final, verbose=False)
print("Final model training complete.")

In [ ]:
# # Save the final model
# import pickle

# training_version = '2.0.7'
# model_filename = f'lantern_xgboost_t{training_version}.pkl'

# with open(model_filename, 'wb') as f:
#     pickle.dump(final_model, f)

# print(f"Model saved to {model_filename}")

### Example false-positives

In [ ]:
import importlib
import data_processing as dp
importlib.reload(dp)
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

all_probs = model.predict_proba(X)[:, 1]
XGB_THRESHOLD = 0.5 ## set cutoff

# ── False Positives (DP1 DIASources flagged as LAGN) ──────────────────────────
df_fp = df[(df['label'] == 0) & (all_probs > XGB_THRESHOLD)]
print(f"False positives: {len(df_fp):,}")
fp_table = Table.from_pandas(df_fp)

# Load full DIASource data for all 3 fields
full_table = dp.load_dp1(fetch_from_server=False, load_all_fields=True, service=dp.service)
full_table = dp.add_engineered_features(full_table)

# Match fp_table to full_table by ra/dec (strip any existing units with np.array)
fp_coords   = SkyCoord(ra=np.array(fp_table['ra'])*u.deg,  dec=np.array(fp_table['dec'])*u.deg)
full_coords = SkyCoord(ra=np.array(full_table['ra'])*u.deg, dec=np.array(full_table['dec'])*u.deg)

idx, sep, _ = fp_coords.match_to_catalog_sky(full_coords)
matched_mask = sep < 0.5 * u.arcsec
fp_full = full_table[idx[matched_mask]]
print(f"Matched {matched_mask.sum():,} / {len(fp_table):,} false positives in full table")

fig_lsst, fig_legacy = dp.create_image_gallery(fp_full, rows=3, cols=2, include_legacy=False)
plt.show()

---
## 4. Survey Scale Analysis

The test set metrics above reflect a ~1:50 class ratio, not the LSST DIASource LAGN to non-LAGN ratio of ~1:37,000. At survey scale, the same classifier will produce far more false positives relative to true positives.

Here we use only invariant quantities like true positive rate (completeness, TPR) and false positive rate (FPR) to prepare a representative completeness/purity curve.

## Survey Population Estimates

We estimate the number of LAGN and non-LAGN DIASources expected in the 10-year LSST survey and in year 2 (when the sky is fully templated). These are rough calculations.

### LAGN DIASources

| Parameter | Value |
|---|---|
| Expected lensed AGN in full survey | 2,658 |
| Visits per sky position (over 10 years) | 800 |
| LAGN detection rate (DIASource per visit per LAGN) | 10% |
| **Total LAGN DIASources (10 yr)** | **212,640** |
| **Total LAGN DIASources (yr 2, 80 visits)** | **21,264** |

### Non-LAGN DIASources

Non-LAGN DIASources are estimated from the LSST DP1 ECDFS (within the wide fast deep survey area).

| Parameter | Value |
|---|---|
| DP1 DIASources (ECDFS) | 551,975 |
| DP1 visits | 855 |
| DP1 field area | 0.785 deg² |
| DIASources / visit / deg² | 822.4 / visit / deg² |
| Full survey area | 18,000 deg² |
| **Total background DIASources (10 yr)** | **~11.8 billion** |
| **Total background DIASources (yr 2)** | **~1.18 billion** |


The ratio of LAGN to non-LAGN DIASources is **~1:56,000**.

### Completeness–Purity Analysis

We sweep the classification threshold and compute the completeness and purity at survey scale using 10-fold stratified cross-validation (for smoothness).

Completeness (TPR) and FPR are can be measured on the training data and then rescaled to yield counts of true positives and false positives:

$$N_\text{LAGN,yr2} = 21,264$$
$$N_\text{non-LAGN,yr2} = 1,184,257,459$$

$$\text{TP} = \text{TPR} \times N_\text{LAGN,yr2}$$
$$\text{FP} = \text{FPR} \times N_\text{non-LAGN,yr2}$$
$$\text{purity} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

The purity is averaged across all 10 folds. We produce two sets of plots: one at the DIASource level, and one at the target level. In each case we plot:
1. **Completeness vs. Purity**
2. **LAGN DIASources vs. Purity** (how many LAGN DIASources detected at each purity level in year 2)

In [ ]:
# these can be rescaled for the true yr1 data, which will cover the DP2 area
# we have been assuming a 1000 sq deg. area for DP2 (which we note will not be fully templated)

lagn_diasources     = 21_264
non_lagn_diasources = 1_184_257_459
N_lenses_yr1 = 2658  #note that this is now actually year 2, when the sky will be fully templated

N_nonlenses_full = df[df['label'] == 0]['lens_id'].nunique()
N_nonlagn_diasources_full = (df['label'] == 0).sum()
avg_diasources_per_nonlens = N_nonlagn_diasources_full / N_nonlenses_full
N_nonlenses_yr1 = int(non_lagn_diasources / avg_diasources_per_nonlens)

print(f"\nLAGN DIASources     (yr 1) : {lagn_diasources:,}")
print(f"Non-LAGN DIASources (yr 1) : {non_lagn_diasources:,}")
 
cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
comp_grid = np.linspace(0, 1, 100)
 
# Store metrics for both levels
fold_purity_sources = []
fold_purity_objects = []
fold_lens_frac = []
fold_nonlens_frac = []
 
for train_idx, val_idx in cv.split(X_train, y_train, groups=groups_train):
    fold_sw = compute_band_weights(y_train.iloc[train_idx], X_train.iloc[train_idx])
 
    fold_model = XGBClassifier(
        n_estimators=300,
        eval_metric='logloss', enable_categorical=True,
        random_state=42, n_jobs=-1, **params,
    )
    fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx],
                   sample_weight=fold_sw, verbose=False)
 
    # Get predicted probabilities and true labels for validation fold
    prob = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    labels = np.array(y_train.iloc[val_idx])
    val_lens_id = np.array(groups_train.iloc[val_idx])
 
    # Sort validation data by descending predicted probability
    order = np.argsort(prob)[::-1]
    labels_sorted = labels[order]
    lens_id_sorted = val_lens_id[order]
 
    # --------------DIASource-level metrics------------------------------
    N_pos_sources = (labels == 1).sum()
    N_neg_sources = (labels == 0).sum()
 
    TP_sources = np.cumsum(labels_sorted == 1)
    FP_sources = np.cumsum(labels_sorted == 0)
 
    tpr_sources = TP_sources / N_pos_sources
    fpr_sources = FP_sources / N_neg_sources
 
    # Scale to survey population
    TP_survey_sources = tpr_sources * lagn_diasources
    FP_survey_sources = fpr_sources * non_lagn_diasources
    purity_sources = TP_survey_sources / (TP_survey_sources + FP_survey_sources).clip(1e-12)
 
    fold_purity_sources.append(np.interp(comp_grid, tpr_sources, purity_sources))
 
    # --------------Unique object-level metrics------------------------------
    N_lenses_fold = len(np.unique(val_lens_id[labels == 1]))
    N_nonlenses_fold = len(np.unique(val_lens_id[labels == 0]))
 
    # Track unique objects recovered as we descend sorted list
    seen_lenses = set()
    seen_nonlenses = set()
    n_lenses_cumul = np.zeros(len(labels_sorted), dtype=int)
    n_nonlenses_cumul = np.zeros(len(labels_sorted), dtype=int)
 
    for j in range(len(labels_sorted)):
        lid = lens_id_sorted[j]
        if labels_sorted[j] == 1:
            seen_lenses.add(lid)
        else:
            seen_nonlenses.add(lid)
        n_lenses_cumul[j] = len(seen_lenses)
        n_nonlenses_cumul[j] = len(seen_nonlenses)
 
    tpr_objects = n_lenses_cumul / max(N_lenses_fold, 1)
    fpr_objects = n_nonlenses_cumul / max(N_nonlenses_fold, 1)
 
    # Scale to survey population
    TP_survey_objects = tpr_objects * N_lenses_yr1
    FP_survey_objects = fpr_objects * N_nonlenses_yr1
    purity_objects = TP_survey_objects / (TP_survey_objects + FP_survey_objects).clip(1e-12)
 
    fold_purity_objects.append(np.interp(comp_grid, tpr_objects, purity_objects))
 
    fold_lens_frac.append(tpr_objects[-1] if len(tpr_objects) > 0 else 0)
    fold_nonlens_frac.append(fpr_objects[-1] if len(fpr_objects) > 0 else 0)
 
# ----------------Aggregate results across folds------------------------------
mean_purity_sources = np.mean(fold_purity_sources, axis=0)
mean_purity_objects = np.mean(fold_purity_objects, axis=0)

In [ ]:
sample_size_yr1  = comp_grid * lagn_diasources
lenses_yr1       = mean_purity_objects * N_lenses_yr1

#### Completeness vs. Purity @ DIASource Level

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
ax.plot(comp_grid, mean_purity_sources, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity (DIASources, 10-fold CV)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

total_alerts_yr1 = sample_size_yr1 / np.clip(mean_purity_sources, 1e-12, None)

fig, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
fig.subplots_adjust(right=0.75)

ax1.plot(mean_purity_sources, sample_size_yr1, color='darkorange', lw=2, label='DIASources from LAGN')
ax1.set_xscale('log')
ax1.set_xlim(1e-4, 1)
ax1.set_xlabel('Purity')
ax1.set_ylabel('Alerts (DIASources) from LAGN (year 1)', color='darkorange')
ax1.tick_params(axis='y', labelcolor='darkorange')
ax1.spines[['top']].set_visible(False)
ax1.set_title('Sample Composition vs. Purity (year 1, DIASources)')
ax1.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)

ax2 = ax1.twinx()
ax2.spines['right'].set_position(('outward', 0))
ax2.plot(mean_purity_sources, total_alerts_yr1, color='slategrey', lw=2, label='Total alerts')
ax2.set_ylabel('Total alerts (DIASources) in sample (year 1)', color='slategrey')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='slategrey')
ax2.spines[['top']].set_visible(False)

# lines = [ax1.get_lines()[0], ax2.get_lines()[0], ax3.get_lines()[0]]
lines = [ax1.get_lines()[0], ax2.get_lines()[0]]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)

plt.show()

#### Completeness vs. Purity @ Unique Object Level

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
ax.plot(comp_grid, mean_purity_objects, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity (Unique Objects, 10-fold CV)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Plot 2: Sample Composition vs. Purity @ Object Level
sample_size_yr1_objects = comp_grid * N_lenses_yr1
total_objects_yr1 = sample_size_yr1_objects / np.clip(mean_purity_objects, 1e-12, None)

fig, ax1 = plt.subplots(figsize=(6, 4), dpi=300)
fig.subplots_adjust(right=0.75)
ax1.plot(mean_purity_objects, sample_size_yr1_objects, color='forestgreen', lw=2, label='Lenses from LAGN')
ax1.set_xscale('log')
ax1.set_xlim(1e-4, 1)
ax1.set_xlabel('Purity')
ax1.set_ylabel('Unique Lenses from LAGN (year 1)', color='forestgreen')
ax1.tick_params(axis='y', labelcolor='forestgreen')
ax1.spines[['top']].set_visible(False)
ax1.set_title('Sample Composition vs. Purity (year 1, unique objects)')
ax1.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)

ax2 = ax1.twinx()
ax2.spines['right'].set_position(('outward', 0))
ax2.plot(mean_purity_objects, total_objects_yr1, color='slategrey', lw=2, label='Total objects')
ax2.set_ylabel('Total unique objects in sample (year 1)', color='slategrey')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='slategrey')
ax2.spines[['top']].set_visible(False)

lines = [ax1.get_lines()[0], ax2.get_lines()[0]]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)
plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
# ax.plot(comp_grid, mean_purity_objects, color='steelblue', lw=2)
# ax.set_yscale('log')
# ax.set_ylim(1e-4, 1)
# ax.set_xlim(0, 1)
# ax.set(xlabel='Completeness', ylabel='Purity',
#        title='Completeness–Purity (Unique Objects, 10-fold CV)')
# ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
# ax.spines[['top', 'right']].set_visible(False)
# plt.tight_layout()
# plt.show()

# Plot 2: Sample Composition vs. Purity @ Object Level
sample_size_yr1_objects = comp_grid * N_lenses_yr1
total_objects_yr1 = sample_size_yr1_objects / np.clip(mean_purity_objects, 1e-12, None)

fig, ax1 = plt.subplots(figsize=(7, 4), dpi=300)
fig.subplots_adjust(right=0.75)
ax1.plot(mean_purity_objects, sample_size_yr1_objects, color='forestgreen', lw=2, label='LAGN')
ax1.set_xscale('log')
ax1.set_xlim(1e-4, 1)
ax1.set_xlabel('Purity')
ax1.set_ylabel('Unique LAGN (Y2)', color='forestgreen')
ax1.tick_params(axis='y', labelcolor='forestgreen')
ax1.spines[['top']].set_visible(False)
ax1.set_title('Sample Composition vs. Purity of Targets (Y2)')
ax1.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)

ax2 = ax1.twinx()
ax2.spines['right'].set_position(('outward', 0))
ax2.plot(mean_purity_objects, total_objects_yr1, color='slategrey', lw=2, label='All targets')
ax2.set_ylabel('Total Unique Targets (Y2)', color='slategrey')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='slategrey')
ax2.spines[['top']].set_visible(False)

lines = [ax1.get_lines()[0], ax2.get_lines()[0]]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
ax.plot(comp_grid, mean_purity_objects, color='steelblue', lw=2)
ax.set_yscale('log')
ax.set_ylim(1e-4, 1)
ax.set_xlim(0, 1)
ax.set(xlabel='Completeness', ylabel='Purity',
       title='Completeness–Purity for Unique Targets (Y2)')
ax.grid(True, which='both', ls='--', lw=0.5, alpha=0.6)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def print_statistics_at_completeness(target_completeness, comp_grid, 
                                      mean_purity_sources, mean_purity_objects_detectable,
                                      mean_purity_objects_total,
                                      lagn_diasources, non_lagn_diasources,
                                      N_lenses_yr1, N_lenses_yr1_detectable, N_nonlenses_yr1,
                                      detection_efficiency):
    """
    Print detailed statistics at a target completeness level.
    Handles DIASource-level, detectable lens-level, and total lens-level metrics.
    """
    # Find the index closest to target completeness
    idx = np.argmin(np.abs(comp_grid - target_completeness))
    actual_completeness = comp_grid[idx]
    
    # Get purities at this completeness
    purity_sources = mean_purity_sources[idx]
    purity_objects_detectable = mean_purity_objects_detectable[idx]
    
    # For total lens completeness, scale the grid
    comp_grid_total = comp_grid * detection_efficiency
    if target_completeness <= detection_efficiency:
        idx_total = np.argmin(np.abs(comp_grid_total - target_completeness))
        actual_completeness_total = comp_grid_total[idx_total]
        purity_objects_total = mean_purity_objects_total[idx_total]
    else:
        actual_completeness_total = None
        purity_objects_total = None
    
    # ---- DIASource-level statistics ----
    lagn_diasources_recovered = actual_completeness * lagn_diasources
    total_diasources_in_sample = lagn_diasources_recovered / purity_sources
    nonlagn_diasources_in_sample = total_diasources_in_sample - lagn_diasources_recovered
    
    # ---- Object-level statistics (DETECTABLE lenses) ----
    unique_lagn_recovered_detectable = actual_completeness * N_lenses_yr1_detectable
    total_objects_in_sample_detectable = unique_lagn_recovered_detectable / purity_objects_detectable
    unique_nonlagn_in_sample_detectable = total_objects_in_sample_detectable - unique_lagn_recovered_detectable
    
    print(f"\n{'='*70}")
    print(f"STATISTICS AT {target_completeness:.1%} COMPLETENESS")
    print(f"{'='*70}")
    print(f"Actual completeness (from grid): {actual_completeness:.1%}")
    
    print(f"\n{'-'*70}")
    print(f"{'DIASOURCE-LEVEL METRICS':<40}")
    print(f"{'-'*70}")
    print(f"{'Purity:':<40} {purity_sources:.4f} ({purity_sources:.4%})")
    print(f"{'LAGN DIASources recovered:':<40} {lagn_diasources_recovered:,.0f}")
    print(f"{'Non-LAGN DIASources in sample:':<40} {nonlagn_diasources_in_sample:,.0f}")
    print(f"{'Total DIASources in sample:':<40} {total_diasources_in_sample:,.0f}")
    print(f"{'Ratio (non-LAGN : LAGN):':<40} {nonlagn_diasources_in_sample/lagn_diasources_recovered:.1f} : 1")
    
    print(f"\n{'-'*70}")
    print(f"{'OBJECT-LEVEL METRICS (Detectable lenses only)':<40}")
    print(f"{'-'*70}")
    print(f"{'Purity:':<40} {purity_objects_detectable:.4f} ({purity_objects_detectable:.4%})")
    print(f"{'Unique LAGN recovered:':<40} {unique_lagn_recovered_detectable:,.0f}")
    print(f"  of {N_lenses_yr1_detectable:,} detectable ({actual_completeness:.1%})")
    print(f"{'Unique non-LAGN in sample:':<40} {unique_nonlagn_in_sample_detectable:,.0f}")
    print(f"{'Total unique objects in sample:':<40} {total_objects_in_sample_detectable:,.0f}")
    print(f"{'Ratio (non-LAGN : LAGN):':<40} {unique_nonlagn_in_sample_detectable/unique_lagn_recovered_detectable:.1f} : 1")
    
    # ---- Object-level statistics (ALL lenses) ----
    if actual_completeness_total is not None:
        unique_lagn_recovered_total = actual_completeness_total * N_lenses_yr1
        total_objects_in_sample_total = unique_lagn_recovered_total / purity_objects_total
        unique_nonlagn_in_sample_total = total_objects_in_sample_total - unique_lagn_recovered_total
        
        print(f"\n{'-'*70}")
        print(f"{'OBJECT-LEVEL METRICS (All lenses including undetectable)':<40}")
        print(f"{'-'*70}")
        print(f"{'Purity:':<40} {purity_objects_total:.4f} ({purity_objects_total:.4%})")
        print(f"{'Unique LAGN recovered:':<40} {unique_lagn_recovered_total:,.0f}")
        print(f"  of {N_lenses_yr1:,} total lenses ({actual_completeness_total:.1%})")
        print(f"  ({unique_lagn_recovered_total:,.0f} of {N_lenses_yr1_detectable:,} detectable = "
              f"{unique_lagn_recovered_total/N_lenses_yr1_detectable:.1%})")
        print(f"{'Unique non-LAGN in sample:':<40} {unique_nonlagn_in_sample_total:,.0f}")
        print(f"{'Total unique objects in sample:':<40} {total_objects_in_sample_total:,.0f}")
        print(f"{'Ratio (non-LAGN : LAGN):':<40} {unique_nonlagn_in_sample_total/unique_lagn_recovered_total:.1f} : 1")
    else:
        print(f"\n{'-'*70}")
        print(f"{'OBJECT-LEVEL METRICS (All lenses including undetectable)':<40}")
        print(f"{'-'*70}")
        print(f"Target completeness {target_completeness:.1%} EXCEEDS detection efficiency!")
        print(f"Maximum possible completeness: {detection_efficiency:.1%}")
        print(f"  ({N_lenses_yr1_detectable:,} of {N_lenses_yr1:,} total lenses)")
        print(f"\nTo see statistics at maximum completeness, use target = {detection_efficiency:.1%}")
    
    print(f"{'='*70}\n")


# Example usage:
print_statistics_at_completeness(
    target_completeness=0.95,
    comp_grid=comp_grid,
    mean_purity_sources=mean_purity_sources,
    mean_purity_objects_detectable=mean_purity_objects,
    mean_purity_objects_total=mean_purity_objects,
    lagn_diasources=lagn_diasources,
    non_lagn_diasources=non_lagn_diasources,
    N_lenses_yr1=N_lenses_yr1,
    N_lenses_yr1_detectable=N_lenses_yr1,
    N_nonlenses_yr1=N_nonlenses_yr1,
    detection_efficiency=0.99
)

# Try a completeness that exceeds detection efficiency
print_statistics_at_completeness(
    target_completeness=0.999,
    comp_grid=comp_grid,
    mean_purity_sources=mean_purity_sources,
    mean_purity_objects_detectable=mean_purity_objects,
    mean_purity_objects_total=mean_purity_objects,
    lagn_diasources=lagn_diasources,
    non_lagn_diasources=non_lagn_diasources,
    N_lenses_yr1=N_lenses_yr1,
    N_lenses_yr1_detectable=N_lenses_yr1,
    N_nonlenses_yr1=N_nonlenses_yr1,
    detection_efficiency=0.99
)